### Imports

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install fuzzywuzzy

In [3]:
# Run
import pandas as pd
import numpy as np
from fuzzywuzzy import fuzz
import os
import tqdm
from copy import deepcopy

/usr/local/lib/python3.12/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


### Data

We assume that the filenames in "path_to_llm_folder" and "path_to_manual_folder" are identical. If not, create a dictionary mapping names from "path_to_llm_folder" to corresponding one in "path_to_manual_folder".

In [4]:
path_to_llm_results = '/content/drive/My Drive/Madagascar birds/llm_test_results.csv'
path_to_manual_results = '/content/drive/My Drive/Madagascar birds/Manual_transcription.csv' # change to transcription file

### Evaluation Functions

#### Scoring

In [5]:
def fuzzy_score_data(df_manual,df_llm,categories):
    diff_df = pd.DataFrame()
    for name in categories:
        new_df = pd.DataFrame()
        new_df[name+'1'] = df_manual[name].astype(str)
        new_df[name+'1'] = new_df[name+'1'].str.lower()
        new_df[name+'2'] = df_llm[name].astype(str)
        new_df[name+'2'] = new_df[name+'2'].str.lower().replace('\n',' ')
        diff_F = new_df.apply(lambda x: fuzz.ratio(x[name+'1'],  x[name+'2']), axis=1)
        diff_df[name+' (Manual)'] = new_df[name+'1']
        diff_df[name+' (LLM)'] = new_df[name+'2']
        diff_df[name+' (Fuzzy)'] = diff_F
    return diff_df

### Looped Code

In [6]:
main_folder_path = '/contents/drive/My Drive/Madagascar birds' # change accordingly

In [7]:
categories = ['RegistrationNumber', 'Genus', 'Species', 'Subspecies', 'Sex', 'Date', 'Locality', 'PlaceName', 'Region', 'Expedition'] # fill in fully

In [8]:


# 1) Load CSVs
df_manual = pd.read_csv(path_to_manual_results)
df_llm = pd.read_csv(path_to_llm_results)
# 2) Calculate scores
diff_df = fuzzy_score_data(df_manual,df_llm,categories)
# 3) Save results
diff_df.insert(0, 'Filename', list(df_llm['Filename']))
diff_df.to_csv('/content/drive/My Drive/Madagascar birds/fuzzy_test_results.csv',index=False)


ValueError: Length of values (1078) does not match length of index (98)

## Fix to align LLM results with Manual transcription for Fuzzy Testing

In [9]:
# 1) Load CSVs
df_manual = pd.read_csv(path_to_manual_results)
df_llm = pd.read_csv(path_to_llm_results)

# Ensure 'Filename' columns are present for merging the LLM test result data set and Manual Transcription result data set
if 'Filename' not in df_manual.columns:
    raise ValueError("df_manual must contain a 'Filename' column for merging.")
if 'Filename' not in df_llm.columns:
    raise ValueError("df_llm must contain a 'Filename' column for merging.")

# Merge the dataframes on 'Filename' to ensure row-wise correspondence
# Using an inner merge to only keep rows present in both dataframes.
merged_df = pd.merge(df_manual, df_llm, on='Filename', how='inner', suffixes=('_manual', '_llm'))

# Now, create two new dataframes from the merged_df that have the structure
# expected by the fuzzy_score_data function. These will have the same length and corresponding rows.

df_manual_aligned = pd.DataFrame()
df_llm_aligned = pd.DataFrame()

for col in categories:
    # Assuming categories match column names, and merged_df has suffixed columns like 'Genus_manual', 'Genus_llm'
    df_manual_aligned[col] = merged_df[f"{col}_manual"]
    df_llm_aligned[col] = merged_df[f"{col}_llm"]

# 2) Calculate scores using the aligned dataframes
diff_df = fuzzy_score_data(df_manual_aligned, df_llm_aligned, categories)

# 3) Save results
# Now, diff_df has the same number of rows as merged_df.
# We can safely insert the 'Filename' column from merged_df.
diff_df.insert(0, 'Filename', merged_df['Filename'].tolist())
diff_df.to_csv('/content/drive/My Drive/Madagascar birds/fuzzy_test_results.csv',index=False)